In [1]:
import os
from collections import Counter

In [2]:
file = 'data/day12.txt'
path = os.path.join(os.getcwd(), file)
with open(path, 'r') as fp:
    lines = [x.strip() for x in fp.readlines()]

In [3]:
class FarmPlanner():
    def __init__(self, lines):
        self.grid = self.get_grid(lines)
        self.regions = self.partition_regions()
        attribs = self.get_attrib()
        self.corners = attribs.pop()
        self.perimeters = attribs.pop()
        
    def __len__(self):
        return len(self.regions)
    
    def get_grid(self, data):
        grid = {}
        for row, line in enumerate(data):
            for col, value in enumerate(line):
                grid[(row, col)] = value
        return grid
        
    def partition_regions(self):
        grid_copy, regions, i = self.grid.copy(), {}, 0
        while grid_copy:
            regions[i] = set()
            plot = list(grid_copy.keys())[0] # check grid.keys[0]
            del grid_copy[plot]
            plant, cull_list = self.grid[plot], [plot]
            while cull_list:
                row, col = cull_list.pop()
                regions[i].add((row, col))
                adjacent = {(row + 1, col), (row - 1, col), 
                            (row, col + 1), (row, col - 1)}
                for new_plot in adjacent:
                    if new_plot in grid_copy and self.grid[new_plot] == plant:
                        del grid_copy[new_plot]
                        cull_list.append(new_plot)
            i += 1
        return regions
    
    def get_attrib(self):
        perimeters, corners = {}, {}
        def adjacent_plots(loc):
            x, y = loc
            return [(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)]
        
        def corner_plots(loc):
            x, y = loc
            return [{(x + 1, y), (x, y + 1)}, {(x, y + 1), (x - 1, y)},
                    {(x - 1, y), (x, y - 1)}, {(x, y - 1), (x + 1, y)}]
        
        for i, region in self.regions.items():
            outside, inside, total_corners, perimeter = set(), set(), 0, 0
            for plot in region:
                for new_plot in adjacent_plots(plot):
                    if new_plot not in region:
                        perimeter += 1
                        outside.add(new_plot)
                        inside.add(plot)
            for plot in outside:
                for plots in corner_plots(plot):
                    if plots.issubset(region):
                        total_corners += 1
            for plot in inside:
                for plots in corner_plots(plot):
                    if plots.isdisjoint(region):
                        x, y = plot
                        x = [num[0] for num in plots if num[0] != x][0]
                        y = [num[1] for num in plots if num[1] != y][0]
                        if (x, y) not in region:
                            total_corners += 1
            perimeters[i] = perimeter
            corners[i] = total_corners
        return [perimeters, corners]

In [4]:
# part one
farm = FarmPlanner(lines)
print(sum(farm.perimeters[i] * len(farm.regions[i]) for i in range(len(farm))))

1400386


In [5]:
# part two
print(sum(farm.corners[i] * len(farm.regions[i]) for i in range(len(farm))))

851994
